In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve

In [ ]:
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

In [ ]:
print("загрузка и подготовка данных...")
df = pd.read_csv('amazon.csv')

In [ ]:
# очистка датасета
def clean_numeric(s):
    s = str(s).replace('₹', '').replace(',', '').replace('%', '').replace('|', '').strip()
    return pd.to_numeric(s, errors='coerce')

In [ ]:
for col in ['discounted_price', 'actual_price', 'discount_percentage', 'rating', 'rating_count']:
    df[col] = df[col].apply(clean_numeric)

In [ ]:
# создаем дополнительные признаки на основе текста и цен
df['price_diff'] = df['actual_price'] - df['discounted_price']
df['review_title_len'] = df['review_title'].astype(str).str.len()
df['review_content_len'] = df['review_content'].astype(str).str.len()
df['about_product_len'] = df['about_product'].astype(str).str.len()
df['review_word_count'] = df['review_content'].astype(str).str.split().str.len()
df['is_popular_category'] = df['category'].str.contains('Computers&Accessories|Electronics', na=False).astype(int)

In [ ]:
df_clean = df.dropna(subset=['rating', 'discounted_price', 'actual_price', 'rating_count'])

In [ ]:
THRESHOLD = 4.0
df_clean['is_good_rating'] = (df_clean['rating'] >= THRESHOLD).astype(int)

In [ ]:
feature_cols = [
    'discounted_price', 'actual_price', 'discount_percentage', 'rating_count',
    'price_diff', 'review_title_len', 'review_content_len', 'about_product_len',
    'review_word_count', 'is_popular_category'
]

In [ ]:
X = df_clean[feature_cols]
y = df_clean['is_good_rating']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
#вмзуализация распределение целевой переменной
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

In [ ]:
sns.histplot(df_clean['rating'], bins=20, kde=True, ax=axes[0], color='skyblue')
axes[0].axvline(THRESHOLD, color='red', linestyle='--', linewidth=2, label=f'порог = {THRESHOLD}')
axes[0].set_title('исходное распределение рейтинга')
axes[0].set_xlabel('рейтинг')
axes[0].legend()

In [ ]:
class_counts = df_clean['is_good_rating'].value_counts()
sns.barplot(
    x=['плохой (<4.0)', f'хороший (>={THRESHOLD})'], 
    y=[class_counts[0], class_counts[1]], 
    ax=axes[1], palette=['lightcoral', 'lightgreen'], edgecolor='black'
)
axes[1].set_title('баланс классов после бинаризации')
axes[1].set_ylabel('количество образцов')

In [ ]:
for i, v in enumerate([class_counts[0], class_counts[1]]):
    axes[1].text(i, v + 10, str(v), ha='center', fontweight='bold')

In [ ]:
plt.tight_layout()
plt.show()

In [ ]:
#боексплоты
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

In [ ]:
sns.boxplot(y=df_clean['discounted_price'], ax=axes[0, 0], color='lightcoral')
axes[0, 0].set_title('discounted_price')

In [ ]:
sns.boxplot(y=df_clean['discount_percentage'], ax=axes[0, 1], color='lightgreen')
axes[0, 1].set_title('discount_percentage')

In [ ]:
sns.boxplot(y=df_clean['rating'], ax=axes[1, 0], color='lightblue')
axes[1, 0].set_title('rating')
axes[1, 0].axhline(THRESHOLD, color='red', linestyle='--', label='порог классификации')
axes[1, 0].legend()

In [ ]:
sns.boxplot(y=df_clean['price_diff'], ax=axes[1, 1], color='plum')
axes[1, 1].set_title('price_diff')

In [ ]:
plt.suptitle('анализ выбросов (метод IQR)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
#корреляционная матрица признаков
plt.figure(figsize=(12, 10))
corr_matrix = df_clean[feature_cols + ['rating']].corr()

In [ ]:
sns.heatmap(
    corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1,
    square=True, linewidths=0.5, cbar_kws={'shrink': 0.8}
)

In [ ]:
plt.title('корреляционная матрица признаков', fontsize=14, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
lr_params = {'C': [0.01, 0.1, 1, 10, 100], 'penalty': ['l1', 'l2'], 'solver': ['liblinear']}
svm_params = {'C': [0.1, 1, 10], 'kernel': ['linear', 'rbf'], 'gamma': ['scale', 0.01, 0.1]}
knn_params = {'n_neighbors': [3, 5, 7, 9, 11, 15], 'weights': ['uniform', 'distance']}

In [ ]:
#логистическая регрессия
best_lr = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42), 
    lr_params, cv=5, scoring='f1', n_jobs=-1
).fit(X_train_scaled, y_train)

In [ ]:
#метд опорных векторов
best_svm = GridSearchCV(
    SVC(probability=True, random_state=42), 
    svm_params, cv=5, scoring='f1', n_jobs=-1
).fit(X_train_scaled, y_train)

In [ ]:
#метод ближайших соседей
best_knn = GridSearchCV(
    KNeighborsClassifier(), 
    knn_params, cv=5, scoring='f1', n_jobs=-1
).fit(X_train_scaled, y_train)

In [ ]:
models = [
    ('logistic regression', best_lr),
    ('svm', best_svm),
    ('knn', best_knn)
]

In [ ]:
results = []
for name, model in models:
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    results.append({
        'model': name,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'roc-auc': roc_auc_score(y_test, y_proba)
    })

In [ ]:
print("\nрезультаты сравнения моделей:")
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

In [ ]:
# строим ROC-кривую для всех моделей
plt.figure(figsize=(10, 8))

In [ ]:
for name, model in models:
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    
    plt.plot(fpr, tpr, linewidth=2, label=f'{name} (auc={auc:.3f})')

In [ ]:
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='случайный классификатор')
plt.xlabel('false positive rate (fpr)')
plt.ylabel('true positive rate (tpr)')
plt.title('roc-кривая: сравнение моделей')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()